In [1]:
import napari
from tifffile import imread
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import trackpy as tp

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [42]:
# Load image

image = imread('/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241206_vascular/D10/Unperturbed/50ms/Denoised_merged/17_100tp_561-30-50ms-1000g_1_conf561_merged.tif')

In [43]:
print(image.dtype, image.shape, np.min(image), np.max(image))

uint16 (100, 2, 1024, 1024) 0 55453


In [44]:
# Loads image mask

image_mask = imread('/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241206_vascular/D10/Unperturbed/ROIs/Cytoplasm/17_100tp_561-30-50ms-1000g_1_conf561_merged_ROIs.tif')

In [45]:
print(image_mask .dtype, image_mask .shape, np.min(image_mask ), np.max(image_mask ))

int32 (1024, 1024) 0 5


In [5]:
# Initializes Napari viewer

viewer = napari.Viewer()

Invalid schema for package 'ome-types', please run 'npe2 validate ome-types' to check for manifest errors.


In [46]:
# Adds image to Napari viewer

viewer.add_image(image)

<Image layer 'image' at 0x186022ca0>

In [47]:
# Adds mask image to Napari viewer

viewer.add_image(image_mask,
                 colormap="gist_earth",
                 contrast_limits=[0,8],
                 opacity=0.2)

<Image layer 'image_mask' at 0x18937eb20>

In [48]:
# Loads detected spots

spots_path = r"/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241206_vascular/D10/Unperturbed/50ms/Spot_detection/Run20250523/Factor4/17_100tp_561-30-50ms-1000g_1_conf561_merged_spots.csv"
spots = pd.read_csv(spots_path)
spots = spots.loc[:, ~spots.columns.str.startswith('Unnamed')]
spots.head()

,frame,roi_id,x,y,bg_denoised,amp_denoised,sigma_y,sigma_x,mean_raw_intensity,mean_actin_intensity,mean_tubulin_intensity
0,0,1,227.701956,203.862732,3273.293370,7946.191820,1.224529,2.042831,9136.75,1176.50,10610.00
1,0,1,225.812431,217.576878,4898.810229,6320.229319,1.146350,0.986797,9858.25,932.25,11840.50
2,0,1,214.312093,230.676015,4105.332735,6258.677843,1.302625,1.079633,6997.00,1791.25,10640.50
3,0,1,230.206590,253.937224,4064.916617,4490.000000,1.888289,1.531397,8180.50,3627.25,12411.50
4,0,1,238.413349,275.265548,5412.956681,6271.040796,1.182479,1.152839,8761.00,1477.00,7683.75


In [49]:
# Creates df that contains xyct axes in the same order as image --> tcxy

spots_tcyxcoord = spots.iloc[:, np.r_[0,3,2]]
spots_tcyxcoord.insert(1, "channel", 0)
spots_tcyxcoord

,frame,channel,y,x
0,0,0,203.862732,227.701956
1,0,0,217.576878,225.812431
2,0,0,230.676015,214.312093
3,0,0,253.937224,230.206590
4,0,0,275.265548,238.413349
...,...,...,...,...
69348,99,0,1006.146331,147.035272
69349,99,0,1005.601255,157.135236
69350,99,0,1006.007350,252.749665
69351,99,0,1011.725266,260.223379


In [50]:
# Creates array from df (necessary to use in the points layer of Napari)

spots_tcyxarray = spots_tcyxcoord.to_numpy()
spots_tcyxarray

array([[   0.        ,    0.        ,  203.862732  ,  227.70195602],
       [   0.        ,    0.        ,  217.57687764,  225.81243127],
       [   0.        ,    0.        ,  230.6760154 ,  214.31209297],
       ...,
       [  99.        ,    0.        , 1006.00735011,  252.74966498],
       [  99.        ,    0.        , 1011.72526565,  260.22337871],
       [  99.        ,    0.        , 1015.61701694,  146.7878227 ]])

In [51]:
# Adds detected spots to points layer

viewer.add_points(spots_tcyxcoord,
                  face_color='#ffffff00',
                  edge_color='magenta',
                  name='spots_log')

<Points layer 'spots_log' at 0x1860a1490>

In [ ]:
# Loads tracks

tracks_path = r"/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D2/Puro10min/Tracking/Run20241220/Factor3Link5/17_100tp_561-60-50ms-1000g_1_conf561_merged_spots_tracks.csv"
tracks = pd.read_csv(tracks_path)
tracks = tracks.loc[:, ~tracks.columns.str.startswith('Unnamed')]
tracks.head()

In [ ]:
# Takes columns "particle", "frame", "x", "y" and sorts into format required by napari for visualization
# Inserts channel information right before y and x coordinate, according to image dimensions

tracks_vis = tracks.iloc[:, np.r_[13,0,3,2]].sort_values(["unique_id", "frame"])
tracks_vis.insert(2, "channel", 0)
tracks_vis.head(20)

In [ ]:
viewer.add_tracks(tracks_vis[['unique_id', 'frame', 'channel', 'y', 'x']], name='tracks', 
                  properties={
                      'id': tracks_vis['unique_id'].values,
                             },
                  color_by='id',
                  colormap='hsv', 
                 )

In [ ]:
# Splits up ROIs

tracks_ROIs = [j for i,j in tracks.groupby("roi_id")]
tracks_ROIs[1].head()

In [ ]:
# Plots tracks for one ROI of an image

fig, ax = plt.subplots()
tp.plot_traj(tracks_ROIs[0], ax=ax)

ax.set_xlim(320,530)
ax.set_ylim(735,690)

plt.show()